# SQL on FHIR — Demo

This notebook demonstrates the **SQL on FHIR** `ViewDefinition/$run` endpoint on a FHIR R4 server.

[SQL on FHIR](https://sql-on-fhir.org) defines a standard way to query FHIR resources using
**ViewDefinitions** — declarative views that map FHIRPath expressions to flat tabular columns.
The `$run` operation evaluates a ViewDefinition and returns rows as a JSON array.

### Three execution modes

| Scenario | ViewDefinition | Resources | Works with |
|----------|---------------|-----------|------------|
| 1 | Inline (`viewResource`) | Inline (`resource` params) | H2 + PostgreSQL |
| 2 | Stored (`viewReference`) | Inline (`resource` params) | H2 + PostgreSQL |
| 3 | Stored (`viewReference`) | Stored (database) | PostgreSQL only |

**Prerequisites:** FHIR server running at `http://localhost:9090` with PostgreSQL,  
and data seeded via `python scripts/seed_and_analyze.py`.

In [ ]:
%pip install requests requests-toolbelt pandas matplotlib

In [ ]:
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
from requests_toolbelt.utils import dump

BASE_URL = "http://localhost:9090/fhir/r4"
HEADERS  = {"Content-Type": "application/fhir+json"}

def http_dump(response: requests.Response) -> None:
    """Print the full raw HTTP request and response, exactly as they travelled on the wire."""
    data = dump.dump_all(response)
    print(data.decode("utf-8", errors="replace"))

## 1. Seeded Data

Preview the 20 synthetic patients and their conditions that were seeded via `scripts/seed_and_analyze.py`.

In [ ]:
resp = requests.get(f"{BASE_URL}/Patient", params={"_count": 20}, headers=HEADERS)
resp.raise_for_status()

entries = resp.json().get("entry", [])

patient_rows = []
for e in entries:
    r = e["resource"]
    name = r.get("name", [{}])[0]
    addresses = r.get("address", [])
    patient_rows.append({
        "id":        r.get("id"),
        "name":      f"{name.get('given', [''])[0]} {name.get('family', '')}".strip(),
        "gender":    r.get("gender"),
        "birthDate": r.get("birthDate"),
        "addresses": len(addresses),
        "city":      ", ".join(a.get("city", "") for a in addresses),
        "zip":       ", ".join(a.get("postalCode", "") for a in addresses),
    })

# Capture full resources for inline use in Scenarios 1 & 2
sample_patients = [e["resource"] for e in entries[:3]]

pd.DataFrame(patient_rows)

In [ ]:
resp = requests.get(f"{BASE_URL}/Condition", params={"_count": 100}, headers=HEADERS)
resp.raise_for_status()

condition_rows = []
for e in resp.json().get("entry", []):
    r = e["resource"]
    coding = r.get("code", {}).get("coding", [{}])[0]
    status = r.get("clinicalStatus", {}).get("coding", [{}])[0]
    condition_rows.append({
        "id":             r.get("id"),
        "patient":        r.get("subject", {}).get("reference"),
        "condition_code": coding.get("code"),
        "condition":      coding.get("display"),
        "status":         status.get("code"),
    })

pd.DataFrame(condition_rows)

## 2. ViewDefinition: Patient IDs + Zip Codes

This ViewDefinition introduces two key SQL on FHIR features:

- **`getResourceKey()`** — built-in that returns the resource's logical ID (more portable than `id`).
- **`forEach`** — iterates over a repeated element (here `address`), producing one row per entry.
  A patient with two addresses appears in two rows; all columns from outer `select` blocks are
  repeated on each row.

```
Patient ──► row 1: id=pt-001, zip=62701  (home address)
        └─► row 2: id=pt-001, zip=46201  (work address)
```

In [ ]:
PATIENT_ZIPS_VIEW = {
    "resourceType": "ViewDefinition",
    "name":   "PatientZips",
    "status": "active",
    "resource": "Patient",
    "select": [
        {
            "column": [{"name": "id", "path": "getResourceKey()"}]
        },
        {
            "forEach": "address",
            "select":  [{"column": [{"name": "zip", "path": "postalCode"}]}]
        }
    ]
}

print(json.dumps(PATIENT_ZIPS_VIEW, indent=2))

### Scenario 1 — Inline ViewDefinition + inline resources

Both the ViewDefinition (`viewResource`) and the FHIR resources (`resource`) are passed directly
in the request body as parameters of a FHIR `Parameters` resource.
The server evaluates FHIRPath expressions in memory — **no database access needed**.

Works with both H2 and PostgreSQL backends.

In [ ]:
payload_s1 = {
    "resourceType": "Parameters",
    "parameter": [
        {"name": "viewResource", "resource": PATIENT_ZIPS_VIEW},
        *[{"name": "resource", "resource": p} for p in sample_patients],
    ]
}

resp_s1 = requests.post(f"{BASE_URL}/ViewDefinition/$run", json=payload_s1, headers=HEADERS)
http_dump(resp_s1)

In [ ]:
pd.DataFrame(resp_s1.json())

### Scenario 2 — Stored ViewDefinition + inline resources

The ViewDefinition is **stored** on the server first (`POST /ViewDefinition`), then referenced
by ID via `viewReference`. Resources are still passed inline.

Useful when the same view is re-used across multiple calls.

In [ ]:
# Store the ViewDefinition
resp_store = requests.post(f"{BASE_URL}/ViewDefinition", json=PATIENT_ZIPS_VIEW, headers=HEADERS)
http_dump(resp_store)

stored_vd_id = resp_store.json()["id"]
print(f"\nStored ViewDefinition id: {stored_vd_id}")

In [ ]:
payload_s2 = {
    "resourceType": "Parameters",
    "parameter": [
        {
            "name": "viewReference",
            "valueReference": {"reference": f"ViewDefinition/{stored_vd_id}"}
        },
        *[{"name": "resource", "resource": p} for p in sample_patients],
    ]
}

resp_s2 = requests.post(f"{BASE_URL}/ViewDefinition/$run", json=payload_s2, headers=HEADERS)
http_dump(resp_s2)

In [ ]:
pd.DataFrame(resp_s2.json())

### Scenario 3 — Stored ViewDefinition + stored resources (database path)

No inline `resource` parameters — the server **transpiles the ViewDefinition to SQL** and
executes it directly against the `PatientTable` in PostgreSQL.

This is the production path: all 20 seeded patients are queried, and patients with two
addresses appear in two rows (because of `forEach`).

> **PostgreSQL only** — returns `501 Not Implemented` on H2.

In [ ]:
payload_s3 = {
    "resourceType": "Parameters",
    "parameter": [
        {
            "name": "viewReference",
            "valueReference": {"reference": f"ViewDefinition/{stored_vd_id}"}
        }
    ]
}

resp_s3 = requests.post(f"{BASE_URL}/ViewDefinition/$run", json=payload_s3, headers=HEADERS)
http_dump(resp_s3)

In [ ]:
df_s3 = pd.DataFrame(resp_s3.json())
print(f"{len(df_s3)} rows ({df_s3['id'].nunique()} unique patients — extras have 2 addresses)")
df_s3

## 3. Analytics — Most Common Conditions

Using the `ConditionAnalysis` ViewDefinition from `scripts/seed_and_analyze.py`,
query **all stored conditions** via the database path and rank them by frequency.

In [ ]:
CONDITION_ANALYSIS_VIEW = {
    "resourceType": "ViewDefinition",
    "name":   "ConditionAnalysis",
    "status": "active",
    "resource": "Condition",
    "select": [{
        "column": [
            {"name": "id",                "path": "id"},
            {"name": "patient_ref",       "path": "subject.reference"},
            {"name": "condition_code",    "path": "code.coding.first().code"},
            {"name": "condition_display", "path": "code.coding.first().display"},
            {"name": "clinical_status",   "path": "clinicalStatus.coding.first().code"},
        ]
    }]
}

payload_analytics = {
    "resourceType": "Parameters",
    "parameter": [{"name": "viewResource", "resource": CONDITION_ANALYSIS_VIEW}]
}

resp_analytics = requests.post(f"{BASE_URL}/ViewDefinition/$run", json=payload_analytics, headers=HEADERS)
http_dump(resp_analytics)

In [ ]:
df_conditions = pd.DataFrame(resp_analytics.json())
df_conditions

In [ ]:
counts = df_conditions["condition_display"].value_counts().rename_axis("condition").reset_index(name="count")
counts

In [ ]:
sorted_counts = counts.sort_values("count")

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(sorted_counts["condition"], sorted_counts["count"], color="steelblue")
ax.bar_label(bars, padding=4)
ax.set_xlabel("Number of patients")
ax.set_title("Most Common Conditions (SQL on FHIR — $run database path)")
ax.set_xlim(0, sorted_counts["count"].max() + 2)
plt.tight_layout()
plt.show()

print(f"\nMost common: {counts.iloc[0]['condition']} ({counts.iloc[0]['count']} patients)")